In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import torch
import openpyxl

import networkx as nx
import seaborn as sns
from scipy.stats import spearmanr
from IPython.display import display, Math

# PyTorch GNN / PyG
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.loader import DataLoader

# Train/Test and utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Logistic Regression and Imputation
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Autoencoder
from torch.utils.data import TensorDataset



Date Created: 01_27_2026

This notebook is a streamlined version of the Tumor GNN workflow: data processing → feature construction → model training → results.
It keeps the core logic while removing extra experimental branches.

### GNN Architecture Comparison (quick view)
| Model | Purpose | Notes |
|---|---|---|
| TumorGNN_Tiny | Classification (UNT vs ENC+PAN) | 1-layer GCN + mean pool + linear head |
| TumorGNN_Reg | Regression (tumor volume) | 2-layer GCN + mean pool + MLP head |
| TumorAE | Denoising tumor×kinase matrix | AE used to rebuild node features |



## 1. Data Processing 

Summary of steps and how the data looks like:
- Data loading and preprocessing (network edges + tumor matrices)
- Build directed kinase → substrate graph
- Compute topological metrics (in/out degree + centrality)
- Merge network metrics with measurement data
- Build node index mapping + edge_index tensor
- Clean tumor activity matrix and align topology features to node order
- Final output: node feature tensors per tumor (activity + topology)



In [ ]:
# 1. File Paths
network_file_path = "data/2024_PhosphoAtlas 2.0_updated KSP network_withHTKAMconnections.xlsx"
measurement_file_path = "data/BRAF_CDK_kinase profile_202500827_JPC2MBetAL_v01.csv" # NOT tumor ave meas scores (dont use)
tumor_samples_file = "data/TumorSample_normalizedKinaseActivities_2025.xlsx"

# 2. Load Sheets
# PA2 NETWORK (kinase–substrate edges) 
df_network = pd.read_excel(network_file_path, sheet_name=0)
df_network = df_network[['KINASE GENE', 'SUBSTRATE GENE']].dropna()

# Measurement data (CD scores) from tumor Excel file, sheet 2
df_meas = pd.read_excel(
    tumor_samples_file,
    sheet_name="AveComp_VEHvsENC+Pan_All192Kin"
)

# Standardize kinase column name
kinase_col = df_meas.columns[0]   # original column name was "Kinases (ALL = w/ ≥1 biol.pept.)"
df_meas["kinases"] = df_meas[kinase_col].astype(str).str.strip().str.upper()

# Apply mapping fixes (same as topology analysis notebook)
name_map = {
    "NME": "NME1",
    "NME1-NME2": "NME1",
    "THAP12": "PRKRIR",
}
df_meas["kinases"] = df_meas["kinases"].replace(name_map)


# Re-clean tumor summary (sheet 1 at idx 0) to get sample_name + Treatment
tumor_summary_raw = pd.read_excel(tumor_samples_file, sheet_name=0)

# Row 2 = header, data from row 3 onward
header = tumor_summary_raw.iloc[2]
df_tumor_summary = tumor_summary_raw.iloc[3:].copy()
df_tumor_summary.columns = header

# Fix typo in the sample name column
df_tumor_summary = df_tumor_summary.rename(columns={"sampple name": "sample_name"})

# Keep only the columns we care about
df_tumor_summary = df_tumor_summary[["sample_name", "Treatment", "Tum_Vol_mm3_at_collection"]].dropna(subset=["sample_name"])

# Tumor sample activity matrix
# Sheet 3 = "AveComp_VEHvsENC+PAN_All192Kin" # tumor activity matrix for all samples
df_tumor_matrix  = pd.read_excel(tumor_samples_file, sheet_name=2)

print("Loaded:")
print("  Network edges:", len(df_network))
print("  Measurement rows:", len(df_meas))
print("  Tumor summary rows:", len(df_tumor_summary))
print("  Tumor matrix rows:", len(df_tumor_matrix))

# Quick checks
print("\nTumor summary size:", len(df_tumor_summary))
print(df_tumor_summary["Treatment"].value_counts())


In [ ]:
# Build a directed graph: kinase -> substrate
G_network = nx.from_pandas_edgelist(
    df_network,
    source="KINASE GENE",
    target="SUBSTRATE GENE",
    create_using=nx.DiGraph()
)

print("Graph built:")
print("  Nodes:", G_network.number_of_nodes())
print("  Edges:", G_network.number_of_edges())



In [ ]:
# Compute degrees and centrality
in_deg  = dict(G_network.in_degree())
out_deg = dict(G_network.out_degree())
in_cent = nx.in_degree_centrality(G_network)

centrality_df = pd.DataFrame({
    "node": list(in_deg.keys()),
    "in_degree": [in_deg[n] for n in in_deg],
    "out_degree": [out_deg[n] for n in in_deg],
    "in_degree_centrality": [in_cent[n] for n in in_deg]
})

centrality_df.head()



In [ ]:
merged_network = df_meas.merge(
    centrality_df,
    left_on="kinases",
    right_on="node",
    how="inner"
)

print("Merged network rows:", len(merged_network))
merged_network.head()



In [ ]:
# Fix node set and build index mapping (for GNN later) 
all_genes = sorted([str(g) for g in G_network.nodes()]) # needs str formating first, sort so the idx is deterministic

gene2idx = {g: i for i, g in enumerate(all_genes)}
idx2gene = {i: g for g, i in gene2idx.items()}

n_nodes = len(all_genes)
print(f"num of unique genes in graph: {n_nodes}")

# Build edge_index tensor (PyG-style)
src_idx = df_network["KINASE GENE"].astype(str).map(gene2idx).to_numpy() # source index
dst_idx = df_network["SUBSTRATE GENE"].astype(str).map(gene2idx).to_numpy() # destination index 

# tensor that stores all edges of our graph
edge_index = torch.tensor(
    np.vstack([src_idx, dst_idx]),
    dtype=torch.long
)

print("edge_index shape:", edge_index.shape)
print(edge_index) # Ex) Theres an edge from 1 -> 103 


In [ ]:
# Clean and align tumor matrix first

# First col = kinase name, second col = num peptides, the rest are tumor samples
tumor_kin_col = df_tumor_matrix.columns[0]

# Standardize kinase names to match our network naming
df_tumor_matrix["kinases"] = (
    df_tumor_matrix[tumor_kin_col]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace(name_map)  # reuse same mapping: NME, NME1-NME2, THAP12... (came from pa1-pa2 notebook)
)

# Set kinases as index so we can .loc by kinase name
df_tumor_matrix = df_tumor_matrix.set_index("kinases")

# Tumor sample columns = everything except (old) name + peptide count
tumor_sample_cols = df_tumor_matrix.columns[2:]  # skip Kinases(w/ > 1) and NumberOfPeptidesPerKinases
print("Number of tumor samples:", len(tumor_sample_cols))
print("Example tumor columns:", list(tumor_sample_cols[:5]))


In [ ]:
# Align centrality metrics to our master node list (all_genes)
centrality_aligned = (
    centrality_df
    .set_index("node")
    .reindex(all_genes)      # rows in same order as gene2idx
    .fillna(0.0)
)

struct_mat = centrality_aligned[["in_degree", "out_degree", "in_degree_centrality"]].to_numpy(dtype=np.float32)
print("struct_mat shape:", struct_mat.shape)  # (n_nodes, 3)

# centrality_df contained topological info about each node and all_genes is our ordered master list of nodes
# print(centrality_df.head())
# print(all_genes[:10])



In [ ]:
def build_tumor_node_features(tumor_col):
    """
    Build node feature matrix X for a single tumor sample.

    Features per node:
      X[i, 0] = tumor-specific activity value for gene i  (from df_tumor_matrix)
      X[i, 1] = in_degree
      X[i, 2] = out_degree
      X[i, 3] = in_degree_centrality
    """
    # Start with zeros
    X = np.zeros((n_nodes, 4), dtype=np.float32)

    # Add structural features (same for all tumors)
    X[:, 1:] = struct_mat  # columns 1,2,3

    # Fill tumor-specific activity for genes that appear in this tumor matrix
    # df_tumor_matrix index = kinases
    if tumor_col not in df_tumor_matrix.columns:
        raise ValueError(f"{tumor_col} not found in tumor matrix columns.")

    # Series: index = kinase name, values = activity for this tumor
    tumor_series = df_tumor_matrix[tumor_col].dropna()

    for g, val in tumor_series.items():
        if g in gene2idx:
            idx = gene2idx[g]        # look up index for gene
            X[idx, 0] = float(val)   # add the activity feature

    return X


In [ ]:
# Hashmap stores {tumor_sample_name : node_feature matrix} for all 44 samples
tumor_feature_tensors = {}

# Loop through all tumor samples and build their node-feature matrix
for col in tumor_sample_cols:
    X_t = build_tumor_node_features(col)  # build matrix
    tumor_feature_tensors[col] = X_t      # append to dict

print(f"Built feature tensors for {len(tumor_feature_tensors)} tumors.")

# Each node_feature matrix is (num_nodes, 4) such that each node has centrality metrics and CD score* (if available)
# [0] tumor-specific activity for this kinase (0 if missing)
# [1] in_degree
# [2] out_degree
# [3] in_degree_centrality
example_key = list(tumor_feature_tensors.keys())[0]
print("Example tumor column:", example_key)
print("Example X_t shape:", tumor_feature_tensors[example_key].shape)


In [ ]:
# Map: sample_name -> raw treatment label ("UNT" or "ENC+PAN")
sample_to_treatment = (
    df_tumor_summary
    .set_index("sample_name")["Treatment"]
    .to_dict()
)

print("Sample→Treatment examples:")
for k in list(sample_to_treatment.keys())[:5]:
    print(k, "->", sample_to_treatment[k])



In [ ]:
# Binary encode function s.t. UNT = 0 and ENC+PAN = 1
def encode_treatment(t):
    if isinstance(t, str) and "ENC+PAN" in t:
        return 1
    if isinstance(t, str) and "UNT" in t:
        return 0
    return np.nan   # anything weird or missing


In [ ]:
tumor_labels = {}

# Encode each tumor sample
for col in tumor_sample_cols:
    t = sample_to_treatment.get(col, None)
    tumor_labels[col] = encode_treatment(t)

# Keep only tumors with a valid 0/1 label
valid_tumors = [c for c in tumor_sample_cols if not np.isnan(tumor_labels[c])]

print(f"Valid labeled tumors: {len(valid_tumors)}")
print("Example:", valid_tumors[0], "→ label", tumor_labels[valid_tumors[0]])


In [ ]:
dataset = []

for col in valid_tumors:

    # (n_nodes, F) torch tensor (F=4 here 3 centrality and 1 CD activity)
    x_t = torch.tensor(tumor_feature_tensors[col], dtype=torch.float32)

    # IMPORTANT: keep y as shape (1,) because PyG batching will stack into (B,)
    y_t = torch.tensor([tumor_labels[col]], dtype=torch.long)

    data = Data(
        x=x_t,
        edge_index=edge_index,
        y=y_t,
        sample_name=col   # optional but ill keep around for now
    )
    dataset.append(data)

print(f"# graphs in dataset: {len(dataset)}")
print(dataset[0])


In [ ]:
# Extract scalar labels from each Data object
all_labels = np.array([d.y.item() for d in dataset])

# Stratified split so UNT / ENC+PAN balance is preserved
train_idx, test_idx = train_test_split(
    np.arange(len(dataset)),
    test_size=0.25,
    random_state=42,
    stratify=all_labels
)

train_dataset = [dataset[i] for i in train_idx]
test_dataset  = [dataset[i] for i in test_idx]

print(f"# train graphs: {len(train_dataset)}")
print(f"# test  graphs: {len(test_dataset)}")


In [ ]:
# PyG DataLoaders
# NOTE: PyG DataLoader knows how to batch graph objects (Data)
batch_size = 8  # small dataset → small batch size

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)


## 2. Helper Functions
LOOCV, visualization, and training/evaluation helpers.



In [ ]:
# FUNCTIONS FOR TRAINING AND EVALUATING GNN
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in loader:
        batch = batch.to(device)

        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)  # (B, num_classes)

        # IMPORTANT: CrossEntropyLoss expects y shape (B,), not (B,1)
        y = batch.y.view(-1)

        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs

        preds = out.argmax(dim=1)
        total_correct += (preds == y).sum().item()
        total_examples += batch.num_graphs

    avg_loss = total_loss / total_examples
    acc = total_correct / total_examples
    return avg_loss, acc


@torch.no_grad()
def eval_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in loader:
        batch = batch.to(device)

        out = model(batch.x, batch.edge_index, batch.batch)
        y = batch.y.view(-1)

        loss = criterion(out, y)

        total_loss += loss.item() * batch.num_graphs
        preds = out.argmax(dim=1)
        total_correct += (preds == y).sum().item()
        total_examples += batch.num_graphs

    avg_loss = total_loss / total_examples
    acc = total_correct / total_examples
    return avg_loss, acc


In [ ]:
def loocv_gnn_classification(
    dataset,
    model_class,
    model_kwargs,
    train_one_epoch,
    eval_model,
    num_epochs=60,
    batch_size=8,
    lr=5e-4,
    weight_decay=1e-3,
    seed=42,
    patience=10,          # early stopping patience
    min_delta=1e-4,       # required improvement
    verbose_every=0
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    n = len(dataset)
    y_true, y_pred, names = [], [], []

    for i in range(n):
        # split
        test_dataset = [dataset[i]]
        train_dataset = [dataset[j] for j in range(n) if j != i]

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False)

        # fresh model each fold
        model = model_class(**model_kwargs).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = nn.CrossEntropyLoss()

        # early stopping state
        best_test_loss = float("inf")
        best_state = None
        no_improve = 0

        for epoch in range(1, num_epochs + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
            test_loss,  test_acc  = eval_model(model, test_loader,  criterion, device)

            # early stopping check
            if test_loss < best_test_loss - min_delta:
                best_test_loss = test_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1

            if verbose_every and (epoch % verbose_every == 0 or epoch == 1):
                print(f"[fold {i+1:02d}/{n}] epoch {epoch:03d} | "
                      f"train_acc={train_acc:.3f} test_acc={test_acc:.3f} test_loss={test_loss:.4f}")

            if no_improve >= patience:
                break

        # load best model for this fold
        if best_state is not None:
            model.load_state_dict(best_state)

        # final prediction on held-out sample
        model.eval()
        batch = next(iter(test_loader)).to(device)
        with torch.no_grad():
            out = model(batch.x, batch.edge_index, batch.batch)  # (1,2)
            pred = out.argmax(dim=1).item()
            true = batch.y.item()

        y_pred.append(pred)
        y_true.append(true)
        names.append(getattr(test_dataset[0], "sample_name", f"sample_{i}"))

    # summary
    acc = accuracy_score(y_true, y_pred)
    print("\nLOOCV accuracy:", acc)
    print("\nConfusion matrix:\n", confusion_matrix(y_true, y_pred))
    print("\nReport:\n", classification_report(y_true, y_pred))

    return {
        "acc": acc,
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "names": np.array(names),
    }


# ===== Call it on Tiny GNN (pick dataset_denoised or dataset) =====

# model_kwargs_tiny = {
#     "in_channels": dataset_denoised[0].x.shape[1],
#     "hidden_dim": 8,
#     "num_classes": 2,
#     "dropout": 0.0,
# }

# results_loocv = loocv_gnn_classification(
#     dataset=dataset_denoised,          # or dataset (raw)
#     model_class=TumorGNN_Tiny,
#     model_kwargs=model_kwargs_tiny,
#     train_one_epoch=train_one_epoch,
#     eval_model=eval_model,
#     num_epochs=60,                     # keep modest
#     batch_size=8,
#     lr=5e-4,
#     weight_decay=1e-3,
#     patience=10,
#     verbose_every=0
# )

# TP, FN, FP, TN 


In [ ]:
def plot_ae_reconstruction(X_df, X_hat_df, kinase="ABL1"):
    """
    Quick visual check: original vs reconstructed CD for one kinase.
    """
    plt.scatter(X_df[kinase], X_hat_df[kinase])
    plt.xlabel("Original CD")
    plt.ylabel("Reconstructed CD")
    plt.title(f"{kinase} reconstruction")



## 3. Models
TinyTumor GNN (classification), Logistic Regression baseline, Tiny Tumor GNN regression, and Autoencoder for denoising.



In [ ]:
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn as nn
import torch.nn.functional as F

class TumorGNN_Tiny(nn.Module):
    def __init__(self, in_channels, hidden_dim=8, num_classes=2, dropout=0.0):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_dim)
        self.dropout = dropout
        self.lin = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)   # (B, hidden_dim)
        out = self.lin(x)                # (B, 2)
        return out


In [ ]:
# ==== TumorGNN for REGRESSION ====

class TumorGNN_Reg(nn.Module):
    def __init__(self, in_channels, hidden_dim):
        super().__init__()

        # GNN encoder
        self.conv1 = GCNConv(in_channels, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        # Regression head
        self.lin1 = nn.Linear(hidden_dim, hidden_dim)
        self.lin2 = nn.Linear(hidden_dim, 1)  # single scalar output

    def forward(self, x, edge_index, batch):
        # message passing
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        # pool nodes -> tumor embedding
        x = global_mean_pool(x, batch)

        # regression head
        x = F.relu(self.lin1(x))
        x = self.lin2(x)

        return x.squeeze(-1)  # shape: (batch_size,)


In [ ]:
# Tumor Autoencoder Class 
class TumorAE(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = 16, dropout: float = 0.2):
        super().__init__()

        # Encoder: input_dim -> latent_dim
        self.enc1 = nn.Linear(input_dim, 128)
        self.enc2 = nn.Linear(128, 32)
        self.enc3 = nn.Linear(32, latent_dim)

        # Decoder: latent_dim -> input_dim (reconstruction)
        self.dec1 = nn.Linear(latent_dim, 32)
        self.dec2 = nn.Linear(32, 128)
        self.dec3 = nn.Linear(128, input_dim)

        self.dropout = dropout

    # Encode tumor CD profile into latent space
    def encode(self, x):
        x = F.relu(self.enc1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.enc2(x))
        z = self.enc3(x)  # latent embedding
        return z

    # Decode latent embedding back to CD space
    def decode(self, z):
        z = F.relu(self.dec1(z))
        z = F.dropout(z, p=self.dropout, training=self.training)
        z = F.relu(self.dec2(z))
        x_hat = self.dec3(z)  # reconstruction
        return x_hat

    # Forward pass: encode + decode
    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z


## 4. Experiments and Results
Train/evaluate the models, run LOOCV, denoise with AE, and rank kinases by saliency.



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = TumorGNN_Tiny(
    in_channels=dataset[0].x.shape[1],
    hidden_dim=8,
    num_classes=2,
    dropout=0.0
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss()

num_epochs = 80
for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss,  test_acc  = eval_model(model, test_loader,  criterion, device)

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.3f}"
        )



In [ ]:
# 1) Build a classic ML matrix: rows=samples (44), cols=kinases (192)
kin_col = df_tumor_matrix.reset_index().columns[0]  # depends on current state
# If df_tumor_matrix is already indexed by 'kinases', do this:
df_lr = df_tumor_matrix.copy()

# build feature matrix
X_df = df_lr[tumor_sample_cols].T  # shape (44, 192); rows are tumor samples
y = np.array([tumor_labels[c] for c in tumor_sample_cols], dtype=int) # target labels

#  Train/test split 
X_train, X_test, y_train, y_test = train_test_split(
    X_df.values, y, test_size=0.25, random_state=42, stratify=y
)

# Impute missing values with 0 (biologically: kinase not detected / inactive)
imputer = SimpleImputer(strategy="constant", fill_value=0)

X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

# Logistic Regression baseline
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_imp, y_train)

train_acc = accuracy_score(y_train, clf.predict(X_train_imp))
test_acc  = accuracy_score(y_test,  clf.predict(X_test_imp))

print("Baseline Logistic Regression:")
print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)



In [ ]:
# 1) Build AE input matrix X: (num_tumors, num_kinases)
# X_df should be shape (44, 192): rows=tumors, cols=kinases
X = X_df.to_numpy()
print("X shape:", X.shape)

# 2) Impute + scale (AE cannot train with NaNs)
imputer = SimpleImputer(strategy="constant", fill_value=0.0)
X_imp = imputer.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(device)

# 3) DataLoader
batch_size = 8
ae_loader = DataLoader(
    TensorDataset(X_tensor),
    batch_size=batch_size,
    shuffle=True
)

# 4) Train AE
ae_model = TumorAE(input_dim=X_tensor.shape[1], latent_dim=16, dropout=0.2).to(device)
ae_optimizer = torch.optim.Adam(ae_model.parameters(), lr=1e-3, weight_decay=1e-5)
ae_criterion = nn.MSELoss()

num_epochs = 300

ae_model.train()
for epoch in range(1, num_epochs + 1):
    total_loss = 0.0

    for (xb,) in ae_loader:
        ae_optimizer.zero_grad()
        x_hat, z = ae_model(xb)
        loss = ae_criterion(x_hat, xb)
        loss.backward()
        ae_optimizer.step()

        total_loss += loss.item() * xb.size(0)

    avg_loss = total_loss / len(ae_loader.dataset)

    if epoch % 25 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | recon_loss={avg_loss:.6f}")

# 5) Reconstruct ALL tumors (denoised, still in scaled space)
ae_model.eval()
with torch.no_grad():
    X_hat_scaled, Z = ae_model(X_tensor)  # both are torch tensors

X_hat_scaled = X_hat_scaled.detach().cpu().numpy()   # (44, 192)
Z = Z.detach().cpu().numpy()                         # (44, 16)

print("X_hat_scaled shape:", X_hat_scaled.shape)
print("Z shape:", Z.shape)

# 6) Unscale back to original CD-ish units
X_hat = scaler.inverse_transform(X_hat_scaled)  # back to imputed CD scale

# 7) Wrap into a DataFrame aligned to X_df
X_hat_df = pd.DataFrame(
    X_hat,
    index=X_df.index,      # tumor names (row order)
    columns=X_df.columns   # kinase names
)

print("X_hat_df preview:")
display(X_hat_df.iloc[:3, :6])

# notice how reconstruction loss (MSE between X, X_hat) between original tumorxkinase matrix X and reconstructed matrix X_hat decreases during training
# reconstruction_loss = (1/N) * Sum(X-X_hat)^2


In [ ]:
plot_ae_reconstruction(X_df, X_hat_df, kinase="ABL1")



In [ ]:
# ==== Build denoised node-feature tensors for the GNN ====

def build_tumor_node_features_denoised(tumor_name):
    """
    Same as build_tumor_node_features, but uses AE-denoised CD values from X_hat_df.
    """
    Xn = np.zeros((n_nodes, 4), dtype=np.float32)

    # structural features (same for all tumors)
    Xn[:, 1:] = struct_mat  # in_degree, out_degree, in_degree_centrality

    # denoised CD for this tumor: (kinase -> value)
    tumor_series = X_hat_df.loc[tumor_name]

    for kinase, val in tumor_series.items():
        if kinase in gene2idx:
            idx = gene2idx[kinase]
            Xn[idx, 0] = float(val)

    return torch.tensor(Xn, dtype=torch.float32)

# Build denoised feature dict
tumor_feature_tensors_denoised = {}
for name in valid_tumors:
    tumor_feature_tensors_denoised[name] = build_tumor_node_features_denoised(name)

print("Built denoised tensors for:", len(tumor_feature_tensors_denoised), "tumors")
example = valid_tumors[0]
print("Example:", example, tumor_feature_tensors_denoised[example].shape)

# ==== Rebuild PyG dataset (denoised) ====

dataset_denoised = []
for name in valid_tumors:
    x_t = tumor_feature_tensors_denoised[name]
    y_t = torch.tensor([tumor_labels[name]], dtype=torch.long)

    dataset_denoised.append(
        Data(x=x_t, edge_index=edge_index, y=y_t, sample_name=name)
    )

print("# graphs in denoised dataset:", len(dataset_denoised))
print(dataset_denoised[0])


In [ ]:
# ==== Train/test split + loaders (denoised dataset) ====

all_labels = np.array([d.y.item() for d in dataset_denoised])

train_idx, test_idx = train_test_split(
    np.arange(len(dataset_denoised)),
    test_size=0.25,
    stratify=all_labels,
    random_state=42
)

train_dataset_d = [dataset_denoised[i] for i in train_idx]
test_dataset_d  = [dataset_denoised[i] for i in test_idx]

train_loader_d = DataLoader(train_dataset_d, batch_size=8, shuffle=True)
test_loader_d  = DataLoader(test_dataset_d,  batch_size=8, shuffle=False)

print("# train graphs:", len(train_dataset_d))
print("# test graphs:", len(test_dataset_d))


# ==== Re-init + train the GNN on denoised features ====

model_d = TumorGNN_Tiny(
    in_channels=dataset_denoised[0].x.shape[1],
    hidden_dim=16,
    num_classes=2,
    dropout=0.0
).to(device)

optimizer_d = torch.optim.Adam(model_d.parameters(), lr=1e-3, weight_decay=1e-4)
criterion_d = nn.CrossEntropyLoss()

# Training GNN on denoised features loop
num_epochs = 100
for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model_d, train_loader_d, optimizer_d, criterion_d, device)
    test_loss,  test_acc  = eval_model(model_d, test_loader_d,  criterion_d, device)

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"[Denoised] Epoch {epoch:03d} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.3f}"
        )



In [ ]:
model_kwargs_tiny = {
    "in_channels": dataset_denoised[0].x.shape[1] if "dataset_denoised" in globals() else dataset[0].x.shape[1],
    "hidden_dim": 8,
    "num_classes": 2,
    "dropout": 0.0,
}

results_loocv = loocv_gnn_classification(
    dataset=dataset_denoised if "dataset_denoised" in globals() else dataset,
    model_class=TumorGNN_Tiny,
    model_kwargs=model_kwargs_tiny,
    train_one_epoch=train_one_epoch,
    eval_model=eval_model,
    num_epochs=60,
    batch_size=8,
    lr=5e-4,
    weight_decay=1e-3,
    patience=10,
    verbose_every=0
)



In [ ]:
# ==== Build tumor-volume regression dataset for GNN ====

# 1) Build mapping: sample_name -> tumor volume
sample_to_volume = (
    df_tumor_summary
    .set_index("sample_name")["Tum_Vol_mm3_at_collection"]
    .dropna()
    .to_dict()
)

# 2) Sanity check: sample overlap
gnn_samples = set(valid_tumors)
volume_samples = set(sample_to_volume.keys())

print("Samples in GNN:", len(gnn_samples))
print("Samples with volume:", len(volume_samples))
print("Overlap:", len(gnn_samples & volume_samples))

# (Optional) inspect mismatches
missing_volume = gnn_samples - volume_samples
if len(missing_volume) > 0:
    print("Missing volume for:", list(missing_volume)[:3])

# 3) Keep only tumors with both features + volume
valid_volume_tumors = sorted(gnn_samples & volume_samples)
print("Final usable tumors:", len(valid_volume_tumors))

# 4) Build PyG dataset (REGRESSION)
dataset_volume = []

for name in valid_volume_tumors:
    x_t = torch.tensor(
        tumor_feature_tensors[name], dtype=torch.float32
    )                       # (num_nodes, F)
    
    y_t = torch.tensor(
        sample_to_volume[name], dtype=torch.float32
    )                       # scalar regression target

    dataset_volume.append(
        Data(
            x=x_t,
            edge_index=edge_index,
            y=y_t,
            sample_name=name
        )
    )

print("# graphs in volume dataset:", len(dataset_volume))
print(dataset_volume[0])


In [ ]:
# === Training Loop for Regression
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TumorGNN_Reg(
    in_channels=dataset_volume[0].x.shape[1],
    hidden_dim=16
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

def train_one_epoch_reg(model, loader):
    model.train()
    total_loss = 0.0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        preds = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(preds, batch.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_reg(model, loader):
    model.eval()
    preds_all, y_all = [], []

    for batch in loader:
        batch = batch.to(device)
        preds = model(batch.x, batch.edge_index, batch.batch)

        preds_all.append(preds.cpu().numpy())
        y_all.append(batch.y.cpu().numpy())

    preds_all = np.concatenate(preds_all)
    y_all = np.concatenate(y_all)

    mse = ((preds_all - y_all) ** 2).mean()
    mae = np.abs(preds_all - y_all).mean()

    return mse, mae


In [ ]:
# ====== LONG TRAINING TIME =======
# This block trains the TumorGNN_Reg with LOOCV 

from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut()
indices = np.arange(len(dataset_volume))

num_epochs = 80
all_mse, all_mae = [], []

for fold, (train_idx, test_idx) in enumerate(loo.split(indices), start=1):
    # Build per-fold datasets (1 held-out sample)
    train_dataset_r = [dataset_volume[i] for i in train_idx]
    test_dataset_r  = [dataset_volume[i] for i in test_idx]

    # Data loaders for this fold
    train_loader_r = DataLoader(train_dataset_r, batch_size=8, shuffle=True)
    test_loader_r  = DataLoader(test_dataset_r,  batch_size=8, shuffle=False)

    # Train for all epochs on the fold's training set
    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch_reg(model, train_loader_r)

    # Evaluate once on the held-out sample
    mse, mae = eval_reg(model, test_loader_r)
    all_mse.append(mse)
    all_mae.append(mae)

    print(f"[Reg][LOOCV] Fold {fold:03d}/{len(indices)} | test_mse={mse:.4f} | test_mae={mae:.4f}")

# Aggregate LOOCV performance
print(f"[Reg][LOOCV] Mean MSE={np.mean(all_mse):.4f} ± {np.std(all_mse):.4f} | Mean MAE={np.mean(all_mae):.4f} ± {np.std(all_mae):.4f}")



In [ ]:
def train_final_gnn(dataset, model_class, model_kwargs, train_one_epoch, num_epochs=80, batch_size=8, lr=5e-4, weight_decay=1e-3, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = model_class(**model_kwargs).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, loader, optimizer, criterion, device)
        if epoch % 10 == 0 or epoch == 1:
            print(f"[FINAL] Epoch {epoch:03d} | train_loss={train_loss:.4f}, train_acc={train_acc:.3f}")

    return model, device


In [ ]:
final_model, device = train_final_gnn(
    dataset=dataset_denoised if "dataset_denoised" in globals() else dataset,
    model_class=TumorGNN_Tiny,
    model_kwargs={
        "in_channels": dataset_denoised[0].x.shape[1] if "dataset_denoised" in globals() else dataset[0].x.shape[1],
        "hidden_dim": 8,
        "num_classes": 2,
        "dropout": 0.0
    },
    train_one_epoch=train_one_epoch,
    num_epochs=60
)


In [ ]:
def rank_kinases_by_gradient_saliency(
    model,
    dataset,
    idx2gene,
    device,
    feature_idx=0,          # 0 = CD/activity column in your (n_nodes, 4)
    use_abs=True,
    aggregate="mean",       # "mean" or "sum"
):
    """
    Gradient saliency on input node features:
    score(node i) = avg_tumors( |dLoss/dx_i| )  (optionally only one feature column)

    Returns a DataFrame with one row per node.
    """
    model = model.to(device)
    model.eval()
    criterion = nn.CrossEntropyLoss()

    n_nodes = dataset[0].x.size(0)
    scores = torch.zeros(n_nodes, device=device)
    n_graphs = 0

    for data in dataset:
        data = data.to(device)

        # Make x require gradients (do NOT modify dataset in-place)
        x = data.x.clone().detach().requires_grad_(True)

        model.zero_grad(set_to_none=True)

        # Forward (single graph)
        out = model(x, data.edge_index, data.batch if hasattr(data, "batch") else torch.zeros(x.size(0), dtype=torch.long, device=device))

        # Ensure y is shape (1,) for single-graph loss
        y = data.y
        if y.dim() == 0:
            y = y.unsqueeze(0)

        loss = criterion(out, y)
        loss.backward()

        # grad wrt x: shape (n_nodes, F)
        g = x.grad  # tensor

        if feature_idx is not None:
            g = g[:, feature_idx]  # (n_nodes,)
        else:
            g = g.mean(dim=1)       # average across features -> (n_nodes,)

        if use_abs:
            g = g.abs()

        # accumulate
        if aggregate == "sum":
            scores += g
        else:
            scores += g  # we'll divide by n_graphs at end for mean

        n_graphs += 1

    if aggregate == "mean":
        scores = scores / max(n_graphs, 1)

    scores = scores.detach().cpu().numpy()

    rank_df = (
        pd.DataFrame({
            "node": np.arange(len(scores)),
            "gene": [idx2gene[i] for i in range(len(scores))],
            "score_grad_saliency": scores,
        })
        .sort_values("score_grad_saliency", ascending=False)
        .reset_index(drop=True)
    )

    return rank_df

# --- Run saliency ranking ---
rank_df_grad = rank_kinases_by_gradient_saliency(
    model=final_model,              # <-- your trained TinyGNN instance (from your clean block)
    dataset=dataset_denoised,
    idx2gene=idx2gene,
    device=device,
    feature_idx=0                   # IMPORTANT: saliency on CD/activity only
)

display(rank_df_grad.head(25))


In [ ]:
# Train final model on ALL tumors (for saliency ranking)
final_model, device = train_final_gnn(
    dataset=dataset_denoised if "dataset_denoised" in globals() else dataset,
    model_class=TumorGNN_Tiny,
    model_kwargs={
        "in_channels": dataset_denoised[0].x.shape[1] if "dataset_denoised" in globals() else dataset[0].x.shape[1],
        "hidden_dim": 8,
        "num_classes": 2,
        "dropout": 0.0
    },
    train_one_epoch=train_one_epoch,
    num_epochs=60
)

rank_df_grad = rank_kinases_by_gradient_saliency(
    model=final_model,
    dataset=dataset_denoised if "dataset_denoised" in globals() else dataset,
    idx2gene=idx2gene,
    device=device,
    feature_idx=0
)

# Kinase-only filter
kinase_set = set(
    df_network["KINASE GENE"]
    .astype(str).str.strip().str.upper()
)

rank_df_kinase = (
    rank_df_grad[rank_df_grad["gene"].isin(kinase_set)]
    .reset_index(drop=True)
)

print("Kinases in graph:", len(kinase_set))
print("Kinases found in ranking DF:", len(rank_df_kinase))

display(rank_df_kinase.head(25))



In [ ]:
def plot_kinase_subgraph(kinase, G=G_network, max_substrates=15, figsize=(10, 7)):
    """
    Draw a clean slide-ready visualization of a kinase and its direct substrates.

    Parameters
    ----------
    kinase        : str   kinase gene name (e.g. "CDK1")
    G             : DiGraph  full PhosphoAtlas network
    max_substrates: int   cap on substrates shown (ranked by out-degree of substrate)
    figsize       : tuple figure size
    """
    kinase = kinase.upper()

    if kinase not in G:
        raise ValueError(f"{kinase} not found in network.")

    # Direct substrates of this kinase
    substrates = list(G.successors(kinase))

    # If too many, keep the most-connected ones (by their own out-degree)
    if len(substrates) > max_substrates:
        substrates = sorted(substrates, key=lambda n: G.out_degree(n), reverse=True)[:max_substrates]

    # Build subgraph
    nodes = [kinase] + substrates
    subG = G.subgraph(nodes).copy()

    # Layout: kinase in center, substrates arranged around it
    pos = nx.shell_layout(subG, nlist=[substrates, [kinase]])

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor("#f9f9f9")
    fig.patch.set_facecolor("#f9f9f9")

    # Node colors and sizes
    node_colors = ["#E8554E" if n == kinase else "#4A90D9" for n in subG.nodes()]
    node_sizes  = [2200 if n == kinase else 900 for n in subG.nodes()]

    nx.draw_networkx_nodes(subG, pos, node_color=node_colors, node_size=node_sizes, ax=ax)
    nx.draw_networkx_edges(
        subG, pos,
        edge_color="#888888",
        arrows=True,
        arrowstyle="-|>",
        arrowsize=18,
        width=1.5,
        connectionstyle="arc3,rad=0.1",
        ax=ax
    )
    nx.draw_networkx_labels(
        subG, pos,
        font_size=9,
        font_color="white",
        font_weight="bold",
        ax=ax
    )

    n_shown = len(substrates)
    n_total = G.out_degree(kinase)
    title = f"{kinase}  →  direct substrates  ({n_shown} of {n_total} shown)"
    ax.set_title(title, fontsize=14, fontweight="bold", pad=14, color="#222222")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
# Example usage — swap in any kinase from your saliency ranking
plot_kinase_subgraph("CDK1")


## 5. Figures

Save publication-style figures for the Results section to `./figures/`.

In [ ]:
from gnn_figures import (
    plot_confusion_matrix,
    plot_top_kinase_saliency,
    plot_tumor_embeddings,
)

plot_confusion_matrix(results_loocv["y_true"], results_loocv["y_pred"])
plot_top_kinase_saliency(rank_df_kinase, top_n=20)
plot_tumor_embeddings(final_model, dataset_denoised, device)


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path                                                      
                                          
def plot_top_kinase_saliency_clean(                                           
    rank_df,                                                                  
    top_n=20,
    gene_col="gene",                                                          
    score_col="score_grad_saliency",                                        
    figsize=(6.5, 7),                       
    bar_color="#3D5A80",                
    title="Top kinases by gradient saliency",
    save_path="figures/kinase_saliency_top20.png",                            
):                                          
    top = rank_df.head(top_n).iloc[::-1]                                      
                                                                            
    fig, ax = plt.subplots(figsize=figsize)                                   
    bars = ax.barh(
        top[gene_col], top[score_col],                                        
        color=bar_color, edgecolor="white", linewidth=0.8,                  
    )
                                                                            
    # Value label at the end of each bar    
    for bar in bars:                                                          
        w = bar.get_width()                                                 
        ax.text(                                                              
            w, bar.get_y() + bar.get_height() / 2,
            f"  {w:.3f}", va="center", fontsize=8, color="#444444",           
        )                                                                   
                                        
    ax.set_xlabel("Gradient saliency  (|∂L / ∂activity|)", fontsize=11)
    ax.set_title(title, fontsize=12, pad=10)                                  
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="y", labelsize=10)                                    
    ax.xaxis.grid(True, linestyle=":", color="#cccccc", alpha=0.7)          
    ax.set_axisbelow(True)                                                    
    ax.margins(x=0.12)  # room for value labels                             
    plt.tight_layout()                                                        
                                            
    if save_path:                                                             
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)           
        plt.savefig(save_path, dpi=300, bbox_inches="tight")                  
        print(f"Saved: {save_path}")                                        
    plt.show()                                                                
                                                                              
                                                                                
plot_top_kinase_saliency_clean(rank_df_kinase, top_n=20)